In [1]:
import pandas as pd
import numpy as np
import os

# --- 경로 설정 (수정됨) ---
# 과제 2의 최종 분석 데이터 경로
VENDOR_ANALYSIS_PATH = "../output/problem2_vendor/problem2_vendor_analysis_base.csv"
# GICS 산업 분류 파일 경로 
GICS_PATH = "../output/gics_clean.csv" 
OUT_DIR = "../output/problem3_sensitivity" # 결과 저장 폴더 (수정됨)

# 반드시 출력 폴더를 생성합니다. (OSError 방지)
try:
    os.makedirs(OUT_DIR, exist_ok=True)
except Exception as e:
    print(f"❌ 오류: 출력 폴더 생성 실패. 경로를 확인하세요: {e}")
    exit()

# GICS 분류 레벨: 'sector'를 기준으로 민감도를 분석합니다.
GICS_LEVELS = ['sector', 'industry_group', 'industry', 'sub_industry']


# --- 1. 데이터 로드 및 시그널 필터링 ---
print("➡️ 1. 데이터 로드 및 시그널 필터링 시작...")

try:
    # 1) 과제 2 분석 데이터 로드 (파일 존재 여부 확인 및 타입 최적화)
    if not os.path.exists(VENDOR_ANALYSIS_PATH):
        raise FileNotFoundError(f"Vendor Analysis 파일이 없습니다: {VENDOR_ANALYSIS_PATH}")
    
    df_analysis = pd.read_csv(
        VENDOR_ANALYSIS_PATH, 
        usecols=['symbol', 'surprise_z', 'return_post_1d', 'return_post_2d'],
        dtype={'surprise_z': np.float32, 'return_post_1d': np.float32, 'return_post_2d': np.float32}
    ).dropna(subset=['surprise_z', 'return_post_1d'])

    # Z-score 통계량 계산
    MEAN_Z_SCORE = df_analysis['surprise_z'].mean() 
    STD_Z_SCORE = df_analysis['surprise_z'].std() 
    
    # Positive Signal인 행만 필터링 
    is_positive_signal = df_analysis['surprise_z'] > MEAN_Z_SCORE + 2 * STD_Z_SCORE
    df_positive_reaction = df_analysis[is_positive_signal].copy()
    
    if df_positive_reaction.empty:
        print("분석: Positive Signal 데이터가 없어 산업별 민감도 분석을 건너뜁니다.")
        exit()

    # 2) GICS 산업 분류 데이터 로드
    if not os.path.exists(GICS_PATH):
        raise FileNotFoundError(f"GICS 파일이 없습니다: {GICS_PATH}")
        
    df_gics = pd.read_csv(GICS_PATH, dtype={c: np.float16 for c in GICS_LEVELS})
    
    # 3) GICS와 Positive Signal 데이터 병합
    df_merged = pd.merge(df_positive_reaction, df_gics, on='symbol', how='left')
    
    # 최종적으로 분석에 필요한 컬럼의 결측치 제거
    df_merged = df_merged.dropna(subset=['sector', 'return_post_1d'])
    
    print(f"✅ 데이터 통합 및 필터링 완료. 분석 대상 데이터 수: {len(df_merged)}개")

except FileNotFoundError as e:
    print(f"❌ 오류: 필요한 파일을 찾을 수 없습니다. 경로를 확인하세요: {e}")
    exit()
except Exception as e:
    print(f"❌ 오류: 데이터 처리 중 예상치 못한 오류 발생: {e}")
    exit()


# --- 2. 산업별 민감도 분석 (Positive Signal 기준) ---

# GICS 'sector' 기준으로 수익률의 평균을 계산
print(f"➡️ GICS '{GICS_LEVELS[0]}' 레벨별 민감도 분석 시작...")

# 그룹핑 및 평균 수익률 계산
df_gics_sensitivity = df_merged.groupby(GICS_LEVELS[0])[['return_post_1d', 'return_post_2d']].mean()
df_gics_sensitivity = df_gics_sensitivity.mul(100).round(3) # 백분율 변환
df_gics_sensitivity = df_gics_sensitivity.rename(columns={'return_post_1d': 'Avg_Return_Post_1D (%)', 'return_post_2d': 'Avg_Return_Post_2D (%)'})


# --- 3. 최종 결과 출력 및 저장 ---

# 결과 저장
OUT_ANALYSIS_FILE = os.path.join(OUT_DIR, "problem3_gics_sensitivity.csv")
try:
    df_gics_sensitivity.to_csv(OUT_ANALYSIS_FILE)
    print(f"\n[OK] 산업별 민감도 분석 결과 저장 완료: {OUT_ANALYSIS_FILE}")
except Exception as e:
    print(f"\n❌ 오류: 분석 결과 저장 실패. {e}")


print("\n" + "="*70)
print(f"과제 3: Positive Signal (Long)에 대한 GICS '{GICS_LEVELS[0]}' 민감도 분석 (Post 1D)")
print("="*70)

# Post 1D 수익률 기준으로 정렬 (가장 수익률이 높은 산업이 가장 민감)
df_gics_sensitivity_pos_sorted = df_gics_sensitivity.sort_values(by='Avg_Return_Post_1D (%)', ascending=False)
print(df_gics_sensitivity_pos_sorted.to_markdown())
print("\n" + "="*70)

# 가장 민감한 산업 추출
if not df_gics_sensitivity_pos_sorted.empty:
    most_sensitive_gics = df_gics_sensitivity_pos_sorted.index[0]
    max_return = df_gics_sensitivity_pos_sorted.iloc[0]['Avg_Return_Post_1D (%)']
    print(f"가장 민감한 산업 ({GICS_LEVELS[0]}): {most_sensitive_gics} (평균 {max_return:.3f}%)")
else:
    print("분석 결과가 비어있습니다.")
print("="*70)

➡️ 1. 데이터 로드 및 시그널 필터링 시작...
✅ 데이터 통합 및 필터링 완료. 분석 대상 데이터 수: 299개
➡️ GICS 'sector' 레벨별 민감도 분석 시작...


/Users/masterj/Documents/GitHub/StockPlay-Data-analysis/5mil/lib/python3.12/site-packages/pandas/io/parsers/c_parser_wrapper.py:234: RuntimeWarning: overflow encountered in cast
  chunks = self._reader.read_low_memory(nrows)


AssertionError: 

In [2]:
import pandas as pd
import numpy as np
import os

# --- 경로 설정 ---
VENDOR_ANALYSIS_PATH = "../output/problem2_vendor/problem2_vendor_analysis_base.csv"
GICS_PATH = "../output/gics_clean.csv" 
OUT_DIR = "../output/problem3_sensitivity" 
os.makedirs(OUT_DIR, exist_ok=True)
GICS_LEVELS = ['sector', 'industry_group', 'industry', 'sub_industry']

# --- 메모리 절약을 위한 샘플링 설정 ---
# NOTE: 커널 정지 방지를 위해, 분석 대상 종목 수를 제한합니다.
# 1000개만 무작위 추출하여 분석을 시도합니다. (상황에 따라 이 값을 줄여야 할 수 있습니다.)
SAMPLE_SIZE = 1000 


print("➡️ 1. 데이터 로드 및 샘플링 시작...")

try:
    # 1) 과제 2 분석 데이터 로드 (메모리 최적화)
    df_analysis = pd.read_csv(
        VENDOR_ANALYSIS_PATH, 
        usecols=['symbol', 'surprise_z', 'return_post_1d', 'return_post_2d'],
        dtype={'surprise_z': np.float32, 'return_post_1d': np.float32, 'return_post_2d': np.float32}
    ).dropna(subset=['surprise_z', 'return_post_1d'])

    # 시그널 필터링
    MEAN_Z_SCORE = df_analysis['surprise_z'].mean() 
    STD_Z_SCORE = df_analysis['surprise_z'].std() 
    is_positive_signal = df_analysis['surprise_z'] > MEAN_Z_SCORE + 2 * STD_Z_SCORE
    df_positive_reaction = df_analysis[is_positive_signal].copy()
    
    if df_positive_reaction.empty:
        print("분석: Positive Signal 데이터가 없어 산업별 민감도 분석을 건너뜁니다.")
        exit()

    # --- 메모리 절약 핵심: 데이터 샘플링 ---
    if len(df_positive_reaction) > SAMPLE_SIZE:
        df_positive_reaction = df_positive_reaction.sample(n=SAMPLE_SIZE, random_state=42).copy()
        print(f"⚠️ 메모리 절약을 위해 Positive Signal 데이터를 {SAMPLE_SIZE}개로 샘플링했습니다.")
        
    # 2) GICS 산업 분류 데이터 로드
    df_gics = pd.read_csv(GICS_PATH, dtype={c: np.float16 for c in GICS_LEVELS})
    
    # 3) GICS와 Positive Signal 데이터 병합
    df_merged = pd.merge(df_positive_reaction, df_gics, on='symbol', how='left')
    df_merged = df_merged.dropna(subset=['sector', 'return_post_1d'])
    
    print(f"✅ 데이터 통합 및 필터링 완료. 최종 분석 대상 데이터 수: {len(df_merged)}개")

except FileNotFoundError as e:
    print(f"❌ 오류: 필요한 파일을 찾을 수 없습니다. 경로를 확인하세요: {e}")
    exit()
except Exception as e:
    print(f"❌ 오류: 데이터 처리 중 예상치 못한 오류 발생: {e}")
    exit()


# --- 4. 산업별 민감도 분석 및 출력 ---
print(f"➡️ GICS '{GICS_LEVELS[0]}' 레벨별 민감도 분석 시작...")

df_gics_sensitivity = df_merged.groupby(GICS_LEVELS[0])[['return_post_1d', 'return_post_2d']].mean()
df_gics_sensitivity = df_gics_sensitivity.mul(100).round(3) 
df_gics_sensitivity = df_gics_sensitivity.rename(columns={'return_post_1d': 'Avg_Return_Post_1D (%)', 'return_post_2d': 'Avg_Return_Post_2D (%)'})

# 결과 저장
OUT_ANALYSIS_FILE = os.path.join(OUT_DIR, "problem3_gics_sensitivity_sampled.csv")
df_gics_sensitivity.to_csv(OUT_ANALYSIS_FILE)

print(f"\n[OK] 산업별 민감도 분석 결과 저장 완료: {OUT_ANALYSIS_FILE}")

print("\n" + "="*70)
print(f"과제 3: Positive Signal (Long)에 대한 GICS '{GICS_LEVELS[0]}' 민감도 분석 (Post 1D)")
print("="*70)

df_gics_sensitivity_pos_sorted = df_gics_sensitivity.sort_values(by='Avg_Return_Post_1D (%)', ascending=False)
print(df_gics_sensitivity_pos_sorted.to_markdown())
print("\n" + "="*70)

if not df_gics_sensitivity_pos_sorted.empty:
    most_sensitive_gics = df_gics_sensitivity_pos_sorted.index[0]
    max_return = df_gics_sensitivity_pos_sorted.iloc[0]['Avg_Return_Post_1D (%)']
    print(f"가장 민감한 산업 ({GICS_LEVELS[0]}): {most_sensitive_gics} (평균 {max_return:.3f}%)")
else:
    print("분석 결과가 비어있습니다.")
print("="*70)

➡️ 1. 데이터 로드 및 샘플링 시작...
✅ 데이터 통합 및 필터링 완료. 최종 분석 대상 데이터 수: 299개
➡️ GICS 'sector' 레벨별 민감도 분석 시작...


/Users/masterj/Documents/GitHub/StockPlay-Data-analysis/5mil/lib/python3.12/site-packages/pandas/io/parsers/c_parser_wrapper.py:234: RuntimeWarning: overflow encountered in cast
  chunks = self._reader.read_low_memory(nrows)


AssertionError: 